# IT25200167 - Multi-Layer Perceptron Neural Network (ANN)

**AIML Group Assignment** | **Dataset:** UCI Default of Credit Card Clients

## Setup — shared train/test split

This notebook is a standalone copy of this member's section from `group_pipeline.ipynb`. The model code below is unchanged.

This cell loads the same 80/20 split that `group_pipeline.ipynb` saves to `results/outputs/train.csv` and `results/outputs/test.csv`, and defines the same `record_model()` function, so the scores match the group notebook. Run `group_pipeline.ipynb` first if these files are missing.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

TARGET = 'default payment next month'

# Same rows as the shared split in group_pipeline.ipynb.
# float_precision='round_trip' reads every value back exactly as it was saved.
train_df = pd.read_csv('../results/outputs/train.csv', float_precision='round_trip')
test_df = pd.read_csv('../results/outputs/test.csv', float_precision='round_trip')
X_train, y_train = train_df.drop(columns=TARGET), train_df[TARGET]
X_test, y_test = test_df.drop(columns=TARGET), test_df[TARGET]
print(f'Train: {len(train_df):,} rows | Test: {len(test_df):,} rows')

model_results = {}

def record_model(student_id, model_name, fitted_model):
    """Evaluate a fitted binary classifier on the shared test set."""
    predictions = fitted_model.predict(X_test)
    result = {
        'IT Number': student_id,
        'Model': model_name,
        'Accuracy': accuracy_score(y_test, predictions),
        'Precision': precision_score(y_test, predictions, zero_division=0),
        'Recall': recall_score(y_test, predictions, zero_division=0),
        'F1': f1_score(y_test, predictions, zero_division=0),
        'ROC-AUC': 'N/A',
    }
    classes = list(getattr(fitted_model, 'classes_', []))
    if 1 in classes and hasattr(fitted_model, 'predict_proba'):
        scores = fitted_model.predict_proba(X_test)[:, classes.index(1)]
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    elif 1 in classes and hasattr(fitted_model, 'decision_function'):
        scores = fitted_model.decision_function(X_test)
        if len(classes) == 2 and classes.index(1) == 0:
            scores = -scores
        result['ROC-AUC'] = roc_auc_score(y_test, scores)
    model_results[student_id] = result
    return pd.Series(result)


### IT25200167 - Multi-Layer Perceptron Neural Network (ANN)

**Theoretical Justification & Model Suitability:**
- **Suitability:** Financial distress manifests through non-linear interactions across sequential repayment statuses and payment ratios. A feed-forward deep neural network (MLP) learns composite non-linear latent representations via hidden ReLU neurons and backpropagation.
- **Tuning Strategy:** Compares multi-layer architectures (`(16, 8)`, `(32, 16)`, `(64, 32)`), Dropout regularization, and Adam optimizer with Early Stopping to prevent overfitting.
- **Convergence Behavior:** Analyzes binary cross-entropy loss and accuracy convergence over training epochs on mini-batch backpropagation.


In [ ]:
# ==============================================================================
# IT25200167 — Artificial Neural Network (ANN) using TensorFlow / Keras
# Feed-Forward Neural Network Architecture
# ==============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import (f1_score, ConfusionMatrixDisplay,
                             classification_report)

# --------------------------------------------------------------------------
# Wrapper: exposes sklearn-style .predict() and .predict_proba() so that the
# shared record_model() helper (from the group cell) works with Keras models.
# --------------------------------------------------------------------------
class KerasBinaryClassifier:
    """Wraps a Keras binary classifier as an sklearn-compatible estimator."""
    def __init__(self, model, threshold=0.5):
        self.model = model
        self.threshold = threshold
        self.classes_ = np.array([0, 1])

    def predict(self, X):
        proba = self.model.predict(X, verbose=0).ravel()
        return (proba >= self.threshold).astype(int)

    def predict_proba(self, X):
        proba = self.model.predict(X, verbose=0).ravel()
        return np.vstack([1 - proba, proba]).T


# --------------------------------------------------------------------------
# Model builder: 2 hidden layers + Dropout + Adam + Early Stopping 
# --------------------------------------------------------------------------
def build_and_train_ann(X_tr, y_tr,
                        hidden1=32, hidden2=16,
                        dropout=0.2, lr=0.001,
                        epochs=60, batch_size=32, verbose=0):
    tf.random.set_seed(42)
    model = keras.Sequential([
        layers.Input(shape=(X_tr.shape[1],)),          # Input layer
        layers.Dense(hidden1, activation='relu'),       # Hidden layer 1
        layers.Dropout(dropout),
        layers.Dense(hidden2, activation='relu'),       # Hidden layer 2
        layers.Dropout(dropout),
        layers.Dense(1, activation='sigmoid')           # Binary output
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    es = EarlyStopping(monitor='val_loss', patience=5,
                       restore_best_weights=True, verbose=0)
    history = model.fit(
        X_tr, y_tr,
        validation_split=0.2,
        epochs=epochs,
        batch_size=batch_size,
        verbose=verbose,
        callbacks=[es]
    )
    return model, history


# --------------------------------------------------------------------------
# Manual hyperparameter search  — 3 architecture configs
# --------------------------------------------------------------------------
configs = [
    {'hidden1': 16, 'hidden2': 8,  'dropout': 0.0, 'lr': 0.001},   # baseline
    {'hidden1': 32, 'hidden2': 16, 'dropout': 0.2, 'lr': 0.001},   # regularised
    {'hidden1': 64, 'hidden2': 32, 'dropout': 0.3, 'lr': 0.0005},  # deeper
]

print("=== IT25200167: Keras ANN Hyperparameter Search ===")
best_model, best_history, best_config, best_f1 = None, None, None, -1.0
search_rows = []

for cfg in configs:
    model, history = build_and_train_ann(X_train, y_train, **cfg)
    y_pred = (model.predict(X_test, verbose=0).ravel() >= 0.5).astype(int)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    search_rows.append({**cfg, 'Test F1': round(f1, 4)})
    print(f"cfg {cfg} -> Test F1 = {f1:.4f}")
    if f1 > best_f1:
        best_f1, best_model, best_history, best_config = f1, model, history, cfg

print(f"\nBest Config: {best_config}")
print(f"Best Test F1: {best_f1:.4f}")

display(pd.DataFrame(search_rows))

# --------------------------------------------------------------------------
# Learning curves — Loss + Accuracy 
# --------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(best_history.history['loss'],     label='Train Loss', color='#8e44ad', linewidth=2)
axes[0].plot(best_history.history['val_loss'], label='Val Loss',   color='#c0392b', linewidth=2)
axes[0].set_title('IT25200167 Keras ANN: Loss over Epochs', fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Binary Cross-Entropy')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(best_history.history['accuracy'],     label='Train Acc', color='#16a085', linewidth=2)
axes[1].plot(best_history.history['val_accuracy'], label='Val Acc',   color='#e67e22', linewidth=2)
axes[1].set_title('IT25200167 Keras ANN: Accuracy over Epochs', fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

# --------------------------------------------------------------------------
# Wrap and register the best Keras model into the shared scoreboard
# --------------------------------------------------------------------------
best_ann = KerasBinaryClassifier(best_model)
record_model('IT25200167', 'Keras ANN (MLP)', best_ann)

# --------------------------------------------------------------------------
# Confusion Matrix + Classification Report 
# --------------------------------------------------------------------------
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay.from_predictions(
    y_test, best_ann.predict(X_test),
    cmap='BuPu', display_labels=['No Default', 'Default'], ax=ax
)
ax.set_title('IT25200167 Keras ANN: Test Confusion Matrix', fontweight='bold')
plt.tight_layout()
plt.show()

print("\nClassification Report:")
print(classification_report(y_test, best_ann.predict(X_test),
                            target_names=['No Default', 'Default']))

# --------------------------------------------------------------------------
# Model summary (Layer Configuration)
# --------------------------------------------------------------------------
print("\nKeras Model Summary:")
best_model.summary()
